# Python practice / Thực hành Python

Tôi lưu bài này trong nhật ký học để ôn lại cách viết và đọc code.

**Nguồn / Source:** Bản Colab `3566_output.ipynb`. Các bài có tên “Bản sao” là tài liệu thực hành tham khảo; tên file không được xem là bằng chứng tác giả.

**Cách dùng / Run:** chạy các cell theo thứ tự. Thư viện được dùng: `numpy`, `pandas`, `scipy`, `scipy.spatial.distance`, `tqdm`, `typing`.

**Execution:** ARCHIVED_SOURCE — bản mã học tập, chưa ghi nhận chạy lại đầy đủ trong đợt này. Output lịch sử được bỏ; hướng dẫn và lỗi cần xử lý nằm trong bảng kỹ thuật.

Learning archive: coursework/tutorial practice, with original code retained for study. Original tutorial references embedded in the source are retained below.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats
from scipy.spatial.distance import cosine
from typing import List, Dict, Tuple

In [ ]:
def load_vocab_dict(path: str) -> Tuple[List[str], Dict[str, int]]:
    """
    Reads a vocabulary list from a file and creates a dictionary mapping each word to its index.

    Args:
        path (str): The file path to the vocabulary list.

    Returns:
        Tuple[List[str], Dict[str, int]]: A tuple containing the list of vocabulary words and a dictionary
                                          mapping each word to its index.
    """
    vocab = open(path).read().strip().split('\n')
    return vocab, {word: idx for idx, word in enumerate(vocab)}

In [ ]:
def read_corpus(path: str) -> List[str]:
    """Reads the corpus from a file, excluding the last empty entry if the file ends with a newline.

    Args:
        path (str): The file path to the corpus.

    Returns:
        List[str]: A list of strings, each representing a line from the file.
    """
    return open(path).read().strip().split('\n')

In [ ]:
def counting(corpus: List[str], V: List[str], V_C: List[str], V_set: Dict[str, int], V_C_set: Dict[str, int]) -> np.ndarray:
    """
    Generates a co-occurrence (counting) matrix from the given corpus, considering specified vocabularies and a window size.

    Args:
        corpus (List[str]): The corpus as a list of sentences.
        V (List[str]): The list of vocabulary words.
        V_C (List[str]): The list of context vocabulary words.
        V_set (Dict[str, int]): A dictionary mapping vocabulary words to their indices.
        V_C_set (Dict[str, int]): A dictionary mapping context vocabulary words to their indices.

    Returns:
        np.ndarray: A 2D NumPy array representing the co-occurrence matrix with dimensions (len(V), len(V_C)).
    """
    # Initialize the matrix to hold word vectors
    C = np.zeros((len(V), len(V_C)), dtype=float)

    for line in tqdm(corpus): # Iterate over each word in the original dataset
        # Append start and end tokens to the sentence
        words = ['<s>'] + line.split(' ') + ['</s>']
        length = len(words)

        for idx, word in enumerate(words): # Iterate over each word in the current sentence
            # Skip '<s>' and '</s>', as they are not real words
            if idx > 0 and idx < length - 1 and word in V_set:
                ### BEGIN SOLUTION
                window_size = 5
                w_idx = V_set[word]
                # Lặp qua các vị trí trong window ±5
                for j in range(max(0, idx - window_size), min(length, idx + window_size + 1)):
                    if j == idx:
                        continue
                    context = words[j]
                    if context not in V_C_set:
                        continue
                    distance = abs(idx - j)
                    weight = 1.05 / distance
                    c_idx = V_C_set[context]
                    # Cập nhật ma trận co-occurrence (symmetric)
                    C[w_idx, c_idx] += weight
                    #C[c_idx, w_idx] += weight

                ### END SOLUTION
    return C

In [ ]:
def eval_word_similarity(C: np.ndarray, V_set: Dict[str, int], path: str) -> float:
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f.readlines()[1:]:
            parts = line.strip().split('\t')
            if len(parts) != 3:
                continue
            w1, w2, score_str = parts
            if w1 in V_set and w2 in V_set:
                score = float(score_str)
                v1, v2 = C[V_set[w1]], C[V_set[w2]]
                sim = 1 - cosine(v1, v2) if np.linalg.norm(v1) > 0 and np.linalg.norm(v2) > 0 else 0.0
                pairs.append((sim, score))

    if not pairs:
        return 0.0

    x, y = zip(*pairs)
    corr = stats.spearmanr(x, y).correlation
    return float(corr) if not np.isnan(corr) else 0.0

In [ ]:
# Read the main vocabulary and its indices from a file,
# creating a list of words (V) and a dictionary mapping words to indices (V_set).
V, V_set = load_vocab_dict('./data/main_words.txt')

# Read the context vocabulary and its indices from a separate file,
# creating a list of context words (V_C) and a dictionary mapping these words to indices (V_C_set).
V_C, V_C_set = load_vocab_dict('./data/context_words.txt')

# Read the corpus from a text file, creating a list where each item represents a document or line in the corpus.
corpus = read_corpus('./data/corpus.txt')

# Generate a co-occurrence (counting) matrix from the corpus using the main and context vocabularies.
C = counting(corpus, V, V_C, V_set, V_C_set)

In [ ]:
### BEGIN PUBLIC TESTS
eval_word_similarity(C, V_set, './data/men.txt')
### BEGIN PUBLIC TESTS

In [ ]:
### BEGIN PUBLIC TESTS
eval_word_similarity(C, V_set, './data/simlex-999.txt')
### BEGIN PUBLIC TESTS

In [ ]:
def improve_C(C, corpus, V, V_set, V_C, V_C_set):
    from collections import Counter
    from scipy.sparse import csr_matrix, lil_matrix
    from scipy.sparse.linalg import svds
    from sklearn.preprocessing import normalize
    import numpy as np

    # Tạo từ điển ánh xạ từ đến chỉ số sử dụng từ vựng ban đầu
    w2i = V_set
    i2w = {i: w for w, i in w2i.items()}
    V_size = len(V)

    # Khởi tạo ma trận đồng xuất hiện
    cooc_matrix = lil_matrix((V_size, V_size), dtype=float)

    # Thiết lập tham số
    window_size = 5

    # Đếm đồng xuất hiện
    for line in tqdm(corpus):
        words = line.strip().split()
        n_words = len(words)

        for i, word in enumerate(words):
            if word not in w2i:
                continue

            word_idx = w2i[word]

            # Xác định phạm vi context
            start = max(0, i - window_size)
            end = min(n_words, i + window_size + 1)

            for j in range(start, end):
                if i == j:
                    continue

                context_word = words[j]
                if context_word not in w2i:
                    continue

                context_idx = w2i[context_word]
                distance = abs(i - j)
                weight = 1.0 / distance  # Trọng số nghịch đảo khoảng cách

                cooc_matrix[word_idx, context_idx] += weight

    # Chuyển đổi sang định dạng CSR để tính toán hiệu quả
    cooc_matrix = cooc_matrix.tocsr()

    # Tính PPMI
    total = cooc_matrix.sum()
    row_sums = np.array(cooc_matrix.sum(axis=1)).flatten()
    col_sums = np.array(cooc_matrix.sum(axis=0)).flatten()

    # Tạo ma trận PPMI
    ppmi_matrix = lil_matrix((V_size, V_size), dtype=float)

    for i in range(V_size):
        for j in range(V_size):
            if cooc_matrix[i, j] > 0:
                p_wc = cooc_matrix[i, j] / total
                p_w = row_sums[i] / total
                p_c = col_sums[j] / total

                pmi = np.log2(p_wc / (p_w * p_c)) if p_w * p_c > 0 else 0
                ppmi = max(pmi, 0)
                ppmi_matrix[i, j] = ppmi

    ppmi_matrix = ppmi_matrix.tocsr()

    # Giảm chiều với SVD
    dim = 300  # Tăng chiều để nắm bắt nhiều thông tin hơn
    U, S, Vt = svds(ppmi_matrix, k=dim)

    # Tạo embeddings từ SVD
    word_embeddings = U @ np.diag(np.sqrt(S))

    # Chuẩn hóa embeddings
    word_embeddings = normalize(word_embeddings, norm='l2', axis=1)

    # Cập nhật ma trận C với các embeddings mới
    for i in range(V_size):
        C[i] = word_embeddings[i]

    return C

### <span style="color:red; font-size:small;">Part #1: {"men": 0.5030654077287334, "simlex-999": 0.19927833121181115}</span>


In [ ]:
# Cải thiện ma trận C
C_improved = improve_C(C.copy(), corpus, V, V_set, V_C, V_C_set)

# Đánh giá hiệu suất
men_score = eval_word_similarity(C_improved, V_set, './data/men.txt')
simlex_score = eval_word_similarity(C_improved, V_set, './data/simlex-999.txt')

print(f"MEN Score: {men_score}")
print(f"SimLex-999 Score: {simlex_score}")

## Ghi chú học tập / Study prompts

- Đầu vào của từng hàm là gì, đầu ra có kiểu và kích thước nào?
- Thay đổi tham số hoặc dữ liệu nhỏ sẽ ảnh hưởng kết quả ra sao?
- Cell nào cần dữ liệu bên ngoài, thư viện hoặc thao tác nhập tay?

Explain the data flow, modify one example, and record the result when revisiting this exercise.